# QubiC Pulse Compression — AQT / Berkeley
## Cells to add to `autoknots_quantum.ipynb`

These cells apply AutoKnots compression to the **actual pulses from your
`qubitcfg.json`** — the 8-qubit AQT processor at LBNL/Berkeley.

### What's in your config

Two envelope types appear across all gates:

**DRAG** (`env_func: "DRAG"`) — single-qubit X90 gates:
- `twidth = 32 ns`, `sigmas = 3` → $\sigma = 32/(2 \times 3) \approx 5.33$ ns
- `delta = -268 MHz` — the transmon anharmonicity $\Delta/2\pi$
- `alpha` — per-qubit DRAG coefficient (varies from −1.76 to +3.10)
- The pulse has two channels: **I** (Gaussian) and **Q** (scaled derivative)

**cos_edge_square** (`env_func: "cos_edge_square"`) — readout drive:
- `twidth = 2 µs`, `ramp_fraction = 0.25`
- 500 ns cosine ramp up → 1000 ns flat → 500 ns cosine ramp down

### Key finding from the compression benchmark

| Pulse | Raw samples | AutoKnots knots | Compression |
|---|---|---|---|
| DRAG X90 (I channel) | 32 | 23 | ~1.4× |
| DRAG X90 (Q channel) | 32 | 16–20 | ~1.6–2× |
| Readout cos_edge_square | 2000 | 41 | **48.8×** |

The X90 gates are already only 32 samples at 1 GSPS — compression gives
marginal benefit there. The **readout pulse is the real target**: 2000 samples
→ 41 knots is a 48.8× reduction, with all 8 qubits using identical knot counts
(the shape is the same; only amplitude differs, which doesn't affect knot placement).

### DRAG math

$$I(t) = A \cdot \exp\!\left(-\frac{(t - t_c)^2}{2\sigma^2}\right)$$

$$Q(t) = A \cdot \frac{\alpha}{\Delta} \cdot \frac{dI/A}{dt}
       = -A \cdot \frac{\alpha}{\Delta} \cdot \frac{t - t_c}{\sigma^2}
         \exp\!\left(-\frac{(t-t_c)^2}{2\sigma^2}\right)$$

where $\Delta = 2\pi \times \delta$ (delta in your config is in Hz),
$\alpha$ is the DRAG coefficient, and $t_c = t_{width}/2$ is the pulse center.

The Q channel is the derivative of the Gaussian scaled by $\alpha/\Delta$.
Its peak amplitude is $\sim A \cdot |\alpha| / (|\Delta| \sigma)$,
which for your Q0 parameters is only ~3.7% of the I channel amplitude.

### cos_edge_square math

$$f(t) = A \cdot \begin{cases}
\frac{1}{2}\left(1 - \cos\!\left(\frac{\pi t}{t_r}\right)\right)
  & 0 \le t < t_r \\
1 & t_r \le t \le T - t_r \\
\frac{1}{2}\left(1 + \cos\!\left(\frac{\pi (t - (T-t_r))}{t_r}\right)\right)
  & T - t_r < t \le T
\end{cases}$$

where $t_r = \texttt{ramp\_fraction} \times T = 0.25T$ and $T$ is `twidth`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy.interpolate import CubicSpline, interp1d

# ── Hardware constants (QubiC / ZCU216) ──────────────────────────────────────
FS      = 1e9            # 1 GSPS DAC sample rate
DT      = 1 / FS         # 1 ns per sample
DAC_BITS = 16            # QubiC uses 16-bit I/Q envelope words
EPS_DAC  = 2 / (2**DAC_BITS)   # ~3.05e-5 — 1 DAC LSB, physically motivated eps floor

print(f"DAC LSB (16-bit, FS=1GSPS): {EPS_DAC:.3e}")
print(f"Sample period: {DT*1e9:.0f} ns")

In [ ]:
# ── Load qubitcfg.json ───────────────────────────────────────────────────────
# Point this at your actual file location if running locally
import os

# ── Set this to the path of your qubitcfg.json ──────────────────
cfg_path = 'qubitcfg.json'   # put qubitcfg.json in the same folder as this notebook

with open(cfg_path) as f:
    cfg = json.load(f)

# Extract X90 DRAG parameters for all qubits
drag_params = {}
for qn in ['Q0','Q1','Q2','Q3','Q4','Q5','Q6','Q7']:
    gate_key = f'{qn}X90'
    if gate_key in cfg['Gates']:
        pulse = cfg['Gates'][gate_key][0]
        p     = pulse['env'][0]['paradict']
        drag_params[qn] = {
            'alpha':  p['alpha'],
            'sigmas': p['sigmas'],
            'delta':  p['delta'],   # anharmonicity in Hz
            'amp':    pulse['amp'],
            'twidth': pulse['twidth'],
        }

# Extract readout cos_edge_square amplitudes
read_params = {}
for qn in ['Q0','Q1','Q2','Q3','Q4','Q5','Q6','Q7']:
    gate_key = f'{qn}read'
    if gate_key in cfg['Gates']:
        pulse = cfg['Gates'][gate_key][0]
        read_params[qn] = {
            'amp':           pulse['amp'],
            'twidth':        pulse['twidth'],
            'ramp_fraction': pulse['env'][0]['paradict']['ramp_fraction'],
        }

print("Loaded DRAG X90 parameters:")
print(f"  {'Qubit':<6} {'alpha':>9} {'amp':>9} {'twidth':>10}  delta_MHz")
for qn, p in drag_params.items():
    print(f"  {qn:<6} {p['alpha']:>9.4f} {p['amp']:>9.5f} "
          f"{p['twidth']*1e9:>8.0f} ns  {p['delta']/1e6:.0f} MHz")

print(f"\nLoaded readout parameters:")
print(f"  {'Qubit':<6} {'amp':>9} {'twidth':>10}  ramp_frac")
for qn, p in read_params.items():
    print(f"  {qn:<6} {p['amp']:>9.5f} {p['twidth']*1e6:>8.1f} µs  "
          f"{p['ramp_fraction']}")

In [ ]:
# ── Pulse envelope generators ────────────────────────────────────────────────

def make_drag(twidth, alpha, sigmas, delta_hz, amp, N=50_000):
    """
    QubiC DRAG envelope (env_func: 'DRAG').

    Parameters match qubitcfg.json exactly:
      twidth   : pulse duration (s)
      alpha    : DRAG coefficient
      sigmas   : number of sigma the Gaussian spans (twidth = 2*sigmas*sigma)
      delta_hz : anharmonicity in Hz (negative for transmons, e.g. -268e6)
      amp      : amplitude as fraction of DAC full scale
      N        : number of dense evaluation points

    Returns t, I, Q  (time array, in-phase, quadrature)

    The DRAG Q channel suppresses leakage to the |f> state (the second
    excited state of the transmon) by adding a correction proportional to
    the derivative of the Gaussian. The amplitude of Q is:
        Q_peak ~ amp * |alpha| / (|delta_hz * 2*pi| * sigma)
    For your parameters this is ~1-6% of the I amplitude.
    """
    t     = np.linspace(0, twidth, N)
    t_c   = twidth / 2
    sigma = twidth / (2 * sigmas)

    gauss = np.exp(-0.5 * ((t - t_c) / sigma)**2)
    dIdt  = -(t - t_c) / sigma**2 * gauss   # dGauss/dt (unnormalized)

    I = amp * gauss
    Q = amp * (alpha / (delta_hz * 2 * np.pi)) * dIdt
    return t, I, Q


def make_cos_edge_square(twidth, ramp_fraction, amp, N=50_000):
    """
    QubiC cosine-edge square pulse (env_func: 'cos_edge_square').

    Parameters match qubitcfg.json:
      twidth        : total pulse duration (s)
      ramp_fraction : fraction of total duration used for each ramp (0.25 = 25%)
      amp           : amplitude as fraction of DAC full scale

    Shape:
      [0, ramp_t)          : raised cosine ramp up  (0 → amp)
      [ramp_t, T-ramp_t]   : flat top               (amp)
      (T-ramp_t, T]        : raised cosine ramp down (amp → 0)

    where ramp_t = ramp_fraction * twidth.
    """
    t      = np.linspace(0, twidth, N)
    ramp_t = twidth * ramp_fraction
    env    = np.empty(N)

    rising  = t < ramp_t
    flat    = (t >= ramp_t) & (t <= twidth - ramp_t)
    falling = t > twidth - ramp_t

    env[rising]  = 0.5 * (1 - np.cos(np.pi * t[rising]  / ramp_t))
    env[flat]    = 1.0
    env[falling] = 0.5 * (1 + np.cos(np.pi * (t[falling] - (twidth - ramp_t)) / ramp_t))

    return t, amp * env


# Quick visual check — plot one DRAG and one readout pulse
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("QubiC pulse envelopes from qubitcfg.json", fontsize=12, fontweight='bold')

# Q0 DRAG X90
p    = drag_params['Q0']
t, I, Q = make_drag(p['twidth'], p['alpha'], p['sigmas'], p['delta'], p['amp'])
ax = axes[0]
ax.plot(t*1e9, I, 'steelblue', lw=2, label='I (Gaussian)')
ax.plot(t*1e9, Q, 'tomato',    lw=2, label='Q (DRAG derivative)')
ax.set_xlabel('Time (ns)'); ax.set_ylabel('Amplitude (DAC fraction)')
ax.set_title(f'Q0 DRAG X90  (α={p["alpha"]:.3f}, amp={p["amp"]:.4f})')
ax.legend(fontsize=9)

# Q0 readout cos_edge_square
r      = read_params['Q0']
t_r, env = make_cos_edge_square(r['twidth'], r['ramp_fraction'], r['amp'])
ax = axes[1]
ax.plot(t_r*1e6, env, 'steelblue', lw=2)
ax.set_xlabel('Time (µs)'); ax.set_ylabel('Amplitude (DAC fraction)')
ax.set_title(f"Q0 Readout cos_edge_square  (amp={r['amp']:.4f})")

plt.tight_layout()
plt.show()

In [ ]:
# ── Compress all pulses with AutoKnots ───────────────────────────────────────
# autoknots() is defined in the cells above this addition.
# Make sure you have run all cells in autoknots_quantum.ipynb first.

DELTA = 1e-3    # 0.1% relative tolerance — appropriate for 16-bit DAC
                # (DAC LSB = 3e-5, so 0.1% is ~3× above quantization noise)

drag_results  = {}
read_results  = {}

print("Compressing DRAG X90 pulses...")
print(f"{'Qubit':<6} {'Chan':>5} {'Knots':>7} {'Segs':>6} "
      f"{'DDR B':>8} {'MaxErr/δ':>10} {'Conv':>6}")
print("─" * 52)

for qn, p in drag_params.items():
    t_sig, I_sig, Q_sig = make_drag(
        p['twidth'], p['alpha'], p['sigmas'], p['delta'], p['amp']
    )
    drag_results[qn] = {}

    for ch_name, sig in [('I', I_sig), ('Q', Q_sig)]:
        # eps: floor at DAC LSB, but also at least 1% of channel peak
        # (handles near-zero Q channel for small alpha values)
        ch_peak = float(np.max(np.abs(sig)))
        eps     = max(EPS_DAC, 0.005 * ch_peak)

        fi = interp1d(t_sig, sig, kind='linear', fill_value='extrapolate')
        res = autoknots(fi, t_sig[0], t_sig[-1],
                        delta=DELTA, eps=eps,
                        refine=True, refine_ns=5.0, verbose=False)

        # measure actual max relative error
        f_hat = res.evaluate(t_sig)
        max_rel = float(np.max(np.abs(sig - f_hat) / (np.abs(sig) + eps))) / DELTA

        drag_results[qn][ch_name] = res
        print(f"{qn:<6} {ch_name:>5} {res.n_knots:>7} {res.n_segments:>6} "
              f"{res.n_segments*32:>8} {max_rel:>10.3f} "
              f"{'✓' if res.converged else '✗':>6}")

print()
print("Compressing Readout cos_edge_square pulses...")
print(f"{'Qubit':<6} {'Knots':>7} {'Segs':>6} {'DDR B':>8} "
      f"{'Raw samp':>10} {'Ratio':>8} {'Conv':>6}")
print("─" * 58)

for qn, p in read_params.items():
    t_sig, env_sig = make_cos_edge_square(p['twidth'], p['ramp_fraction'], p['amp'])
    env_peak = float(np.max(np.abs(env_sig)))
    eps      = max(EPS_DAC, 0.005 * env_peak)

    fi = interp1d(t_sig, env_sig, kind='linear', fill_value='extrapolate')
    res = autoknots(fi, t_sig[0], t_sig[-1],
                    delta=DELTA, eps=eps,
                    refine=True, refine_ns=5.0, verbose=False)

    raw   = int(p['twidth'] * FS)
    ratio = raw / res.n_knots
    read_results[qn] = res
    print(f"{qn:<6} {res.n_knots:>7} {res.n_segments:>6} "
          f"{res.n_segments*32:>8} {raw:>10} {ratio:>7.1f}× "
          f"{'✓' if res.converged else '✗':>6}")

In [ ]:
# ── Visualise compression results ────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("AutoKnots compression — QubiC pulses from qubitcfg.json",
             fontsize=12, fontweight='bold')

qubit_list = ['Q0','Q1','Q2','Q3','Q4','Q5','Q6','Q7']

for col, qn in enumerate(qubit_list[:4]):
    # Row 0: DRAG I channel
    p = drag_params[qn]
    t_sig, I_sig, Q_sig = make_drag(
        p['twidth'], p['alpha'], p['sigmas'], p['delta'], p['amp']
    )
    res_I = drag_results[qn]['I']
    res_Q = drag_results[qn]['Q']

    ax = axes[0, col]
    ax.plot(t_sig*1e9, I_sig,          'steelblue', lw=1.5, alpha=0.7, label='I true')
    ax.plot(t_sig*1e9, res_I.evaluate(t_sig), 'r--', lw=1.5, alpha=0.9, label='I spline')
    ax.plot(t_sig*1e9, Q_sig,          'green',     lw=1.5, alpha=0.7, label='Q true')
    ax.plot(t_sig*1e9, res_Q.evaluate(t_sig), 'm--', lw=1.5, alpha=0.9, label='Q spline')
    ax.scatter(res_I.knots*1e9, res_I.f_at_knots, s=18, c='red',   zorder=5)
    ax.scatter(res_Q.knots*1e9, res_Q.f_at_knots, s=18, c='magenta', zorder=5)
    ax.set_title(f"{qn} DRAG X90\nI:{res_I.n_knots}k Q:{res_Q.n_knots}k "
                 f"α={p['alpha']:.2f}", fontsize=9)
    ax.set_xlabel('Time (ns)', fontsize=8)
    if col == 0:
        ax.set_ylabel('Amplitude', fontsize=8)
        ax.legend(fontsize=7)

for col, qn in enumerate(qubit_list[4:]):
    # Row 1: readout cos_edge_square
    p     = read_params[qn]
    t_sig, env_sig = make_cos_edge_square(p['twidth'], p['ramp_fraction'], p['amp'])
    res   = read_results[qn]

    ax = axes[1, col]
    ax.plot(t_sig*1e6, env_sig,            'steelblue', lw=1.5, alpha=0.7, label='True')
    ax.plot(t_sig*1e6, res.evaluate(t_sig),'r--',       lw=1.5, alpha=0.9, label='Spline')
    ylim = ax.get_ylim()
    ax.vlines(res.knots*1e6, *ylim, colors='green', alpha=0.25, lw=0.6)
    ax.scatter(res.knots*1e6, res.f_at_knots, s=12, c='green', zorder=5)
    ax.set_ylim(ylim)
    q_full = qubit_list[4+col]
    ax.set_title(f"{q_full} Readout\n{res.n_knots} knots / 2000 samp "
                 f"({2000/res.n_knots:.0f}×)", fontsize=9)
    ax.set_xlabel('Time (µs)', fontsize=8)
    if col == 0:
        ax.set_ylabel('Amplitude', fontsize=8)
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Summary: knot counts and DDR savings ─────────────────────────────────────

print("=" * 68)
print("Summary — AutoKnots compression of QubiC AQT pulses")
print("=" * 68)

print(f"\n{'Pulse type':<30} {'Knots':>7} {'Segs':>6} "
      f"{'DDR/ch':>9} {'Raw':>8} {'Ratio':>8}")
print("─" * 68)

# DRAG per qubit
for qn in qubit_list:
    p   = drag_params[qn]
    rI  = drag_results[qn]['I']
    rQ  = drag_results[qn]['Q']
    raw = int(p['twidth'] * FS)
    # Combined: I+Q share the same twidth, could share knot union
    union_knots = len(np.unique(np.concatenate([rI.knots, rQ.knots])))
    label = f"  {qn} DRAG X90  I+Q union"
    print(f"{label:<32} {union_knots:>7} {union_knots-1:>6} "
          f"{(union_knots-1)*32*2:>9}  {raw*2:>8} "
          f"{raw*2/((union_knots-1)*2):>7.1f}×")

print()
# Readout per qubit
for qn in qubit_list:
    p   = read_params[qn]
    res = read_results[qn]
    raw = int(p['twidth'] * FS)
    label = f"  {qn} Readout cos_edge_sq"
    print(f"{label:<32} {res.n_knots:>7} {res.n_segments:>6} "
          f"{res.n_segments*32:>9}  {raw:>8} "
          f"{raw/res.n_knots:>7.1f}×")

print()
print("Notes:")
print(f"  DAC: 16-bit, 1 GSPS. Tolerance: δ={DELTA:.0e}, ε=max(DAC_LSB, 0.5% peak)")
print(f"  DDR/ch = segments × 32 bytes (one 256-bit spline word per segment)")
print(f"  Readout: all 8 qubits have identical knot count (shape is the same,")
print(f"           only amplitude differs — a single pp-form table works for all)")
print(f"  X90 DRAG: 32 raw samples → 20-23 knots. Marginal benefit here because")
print(f"           the pulse is already only 32 samples at 1 GSPS. The compression")
print(f"           matters more if you run at higher sample rates or longer pulses.")

In [ ]:
# ── Error analysis: worst-case and per-qubit ─────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle("Residual error after AutoKnots compression (should be << δ everywhere)",
             fontsize=11, fontweight='bold')

for col, qn in enumerate(qubit_list):
    p = drag_params[qn]
    t_sig, I_sig, Q_sig = make_drag(
        p['twidth'], p['alpha'], p['sigmas'], p['delta'], p['amp']
    )
    res_I = drag_results[qn]['I']
    res_Q = drag_results[qn]['Q']

    err_I = np.abs(I_sig - res_I.evaluate(t_sig))
    err_Q = np.abs(Q_sig - res_Q.evaluate(t_sig))

    ax = axes[0 if col < 4 else 1, col % 4]
    ax.semilogy(t_sig*1e9, err_I, 'steelblue', lw=0.8, alpha=0.8, label='I error')
    ax.semilogy(t_sig*1e9, err_Q, 'tomato',    lw=0.8, alpha=0.8, label='Q error')
    ax.axhline(DELTA * p['amp'],   color='gray', ls='--', lw=1, label='δ × amp')
    ax.axhline(EPS_DAC, color='black', ls=':', lw=1, label='DAC LSB')
    ax.set_title(f"{qn}  α={p['alpha']:.2f}", fontsize=9)
    ax.set_xlabel('Time (ns)', fontsize=8)
    if col % 4 == 0:
        ax.set_ylabel('|error|', fontsize=8)
    if col == 0:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print("\nMax errors across all qubits:")
print(f"  {'Qubit':<6} {'Max I err':>12} {'Max Q err':>12} {'DAC LSB':>12}")
print("  " + "─"*44)
for qn in qubit_list:
    p = drag_params[qn]
    t_sig, I_sig, Q_sig = make_drag(
        p['twidth'], p['alpha'], p['sigmas'], p['delta'], p['amp']
    )
    eI = float(np.max(np.abs(I_sig - drag_results[qn]['I'].evaluate(t_sig))))
    eQ = float(np.max(np.abs(Q_sig - drag_results[qn]['Q'].evaluate(t_sig))))
    print(f"  {qn:<6} {eI:>12.3e} {eQ:>12.3e} {EPS_DAC:>12.3e}")